In [4]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import classification_report, f1_score, accuracy_score, hamming_loss
import mlflow
from pathlib import Path
from IPython.display import display, Markdown

# ==========================================
# 1. SETUP PATH & MLFLOW
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("08_Performance_Evaluation")

print("⏳ 1. Memuat Data Testing (Unseen Data) dan Model Final...")

# 1a. Memuat Data Ujian (20% Test Data yang murni)
test_df = pd.read_csv(root_path / "Data/split/test_data.csv")

# 35 Fitur yang terpilih oleh MFO
selected_features = ['TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 
                     'VCL1', 'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL11', 'VCL12', 
                     'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education', 'urban', 'gender', 'engnat', 'age', 
                     'religion', 'orientation', 'race', 'voted', 'married']

target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

X_test = test_df[selected_features]
Y_test = test_df[target_cols]

# 1b. Memuat Model Final XGBoost
model_path = root_path / "models" / "multilabel_xgboost_mfo.pkl"
with open(model_path, "rb") as f:
    model = pickle.load(f)

with mlflow.start_run(run_name="Evaluate_Unseen_Test_Data"):
    # ==========================================
    # 2. PREDIKSI DATA UJIAN
    # ==========================================
    print("⏳ 2. Memprediksi 3 Kondisi Mental secara bersamaan...")
    Y_pred = model.predict(X_test)
    
    # ==========================================
    # 3. MENGHITUNG METRIK MULTI-LABEL
    # ==========================================
    print("⏳ 3. Menghitung Metrik Evaluasi Khusus Multi-label...")
    
    # A. Exact Match Ratio (Subset Accuracy)
    # Seberapa sering model menebak KETIGA label dengan benar 100% secara bersamaan
    exact_match = accuracy_score(Y_test, Y_pred)
    
    # B. Hamming Loss
    # Rasio kesalahan penebakan per individu label. Semakin mendekati 0 semakin baik.
    h_loss = hamming_loss(Y_test, Y_pred)
    
    # C. Macro & Micro F1-Score
    macro_f1 = f1_score(Y_test, Y_pred, average='macro')
    micro_f1 = f1_score(Y_test, Y_pred, average='micro')
    
    # D. Laporan Klasifikasi per Label
    class_report_str = classification_report(Y_test, Y_pred, target_names=['Depression', 'Anxiety', 'Stress'])
    
    # ==========================================
    # 4. LOGGING MLFLOW & OUTPUT
    # ==========================================
    mlflow.log_metric("test_exact_match", exact_match)
    mlflow.log_metric("test_hamming_loss", h_loss)
    mlflow.log_metric("test_macro_f1", macro_f1)
    mlflow.log_metric("test_micro_f1", micro_f1)
    mlflow.log_text(class_report_str, "classification_report.txt")
    
    summary = f"""
    ### 🎯 Tahap 08: Hasil Evaluasi Akhir (Unseen Test Data)
    ---
    Model telah diuji menggunakan data testing murni sebanyak **{len(X_test):,} responden remaja** yang belum pernah dilihat sebelumnya.
    
    **Metrik Evaluasi Multi-label Utama:**
    * **Macro F1-Score:** **{macro_f1:.4f}** (Rata-rata performa model pada ketiga kondisi mental).
    * **Micro F1-Score:** **{micro_f1:.4f}** (Performa keseluruhan berdasarkan agregat total prediksi benar).
    * **Hamming Loss:** **{h_loss:.4f}** (Artinya, dari semua total label yang ditebak, tingkat kesalahannya hanya sekitar {h_loss*100:.2f}%).
    * **Exact Match Ratio:** **{exact_match:.4f}** (Persentase AI menebak kombinasi 3 label dengan 100% sempurna tanpa meleset satupun).
    
    **Laporan Klasifikasi Detail per Kondisi Mental:**
    ```text
    {class_report_str}
    ```
    ✅ *Evaluasi selesai. Metrik telah tersimpan di MLflow.*
    """
    display(Markdown(summary))
    print(class_report_str) # Menampilkan juga di terminal jika pakai .py

⏳ 1. Memuat Data Testing (Unseen Data) dan Model Final...
⏳ 2. Memprediksi 3 Kondisi Mental secara bersamaan...
⏳ 3. Menghitung Metrik Evaluasi Khusus Multi-label...


d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average


    ### 🎯 Tahap 08: Hasil Evaluasi Akhir (Unseen Test Data)
    ---
    Model telah diuji menggunakan data testing murni sebanyak **5,436 responden remaja** yang belum pernah dilihat sebelumnya.
    
    **Metrik Evaluasi Multi-label Utama:**
    * **Macro F1-Score:** **0.7851** (Rata-rata performa model pada ketiga kondisi mental).
    * **Micro F1-Score:** **0.7855** (Performa keseluruhan berdasarkan agregat total prediksi benar).
    * **Hamming Loss:** **0.2679** (Artinya, dari semua total label yang ditebak, tingkat kesalahannya hanya sekitar 26.79%).
    * **Exact Match Ratio:** **0.5399** (Persentase AI menebak kombinasi 3 label dengan 100% sempurna tanpa meleset satupun).
    
    **Laporan Klasifikasi Detail per Kondisi Mental:**
    ```text
                  precision    recall  f1-score   support

  Depression       0.87      0.71      0.78      3842
     Anxiety       0.88      0.72      0.79      3995
      Stress       0.85      0.72      0.78      3304

   micro avg       0.87      0.72      0.79     11141
   macro avg       0.87      0.72      0.79     11141
weighted avg       0.87      0.72      0.79     11141
 samples avg       0.54      0.55      0.53     11141

    ```
    ✅ *Evaluasi selesai. Metrik telah tersimpan di MLflow.*
    

              precision    recall  f1-score   support

  Depression       0.87      0.71      0.78      3842
     Anxiety       0.88      0.72      0.79      3995
      Stress       0.85      0.72      0.78      3304

   micro avg       0.87      0.72      0.79     11141
   macro avg       0.87      0.72      0.79     11141
weighted avg       0.87      0.72      0.79     11141
 samples avg       0.54      0.55      0.53     11141

